In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")
sns.set_theme(style="whitegrid", palette="deep")

print("✓ Ambiente preparado.")

✓ Ambiente preparado.


In [ ]:
import os
os.environ["DISABLE_COLAB_CACHE"] = "true" # baixar direto do kaggle
import kagglehub

pasta = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")
print("Baixado em:", pasta)
DADOS = os.path.join(pasta, "RAW_recipes.csv")

try:
    df_original = pd.read_csv(DADOS)
    assert df_original.shape == (231637, 12)
except Exception as erro:
    raise RuntimeError(
        "Não foi possível carregar a base."
    ) from erro

print("✓ Dados carregados com sucesso!")
print(f"Fonte: {DADOS}")
print(f"Shape: {df_original.shape[0]} linhas × {df_original.shape[1]} colunas")
print("\nPrimeiras linhas da base recebida:")
display(df_original.head())

In [ ]:
# tipos
df_original.info()

In [ ]:
# ausencias
display(df_original.isna().sum())

In [ ]:
# Q6: o id é único?
print("id é único?", df_original["id"].is_unique)
print("ids repetidos:", df_original["id"].duplicated().sum())

In [ ]:
# Q7: receitas repetidas (mesmo nome e mesmo autor)
duplicadas = df_original.duplicated(subset=["name", "contributor_id"]).sum()
print("Duplicadas por nome + autor:", duplicadas)

df_original[df_original.duplicated(subset=["name", "contributor_id"], keep=False)].sort_values("name")

In [ ]:
# Q8: tempo de preparo, procurando zeros e valores absurdos
display(df_original["minutes"].describe())
print("Receitas com 0 minutos:", (df_original["minutes"] == 0).sum())

In [ ]:
# Q9: tempos muito altos (mais de 1 dia de preparo)
print("Receitas com mais de 1 dia (1440 min):", (df_original["minutes"] > 1440).sum())
display(df_original.nlargest(5, "minutes")[["name", "minutes"]])

In [ ]:
# Q10: n_steps e n_ingredients dentro do domínio do dicionário
print("n_steps = 0:", (df_original["n_steps"] == 0).sum())
print("n_ingredients < 1:", (df_original["n_ingredients"] < 1).sum())
display(df_original[["n_steps", "n_ingredients"]].describe())

In [ ]:
# Q11: datas válidas e dentro de 1999–2018
datas = pd.to_datetime(df_original["submitted"], errors="coerce")
print("Datas inválidas:", datas.isna().sum())
print("Menor data:", datas.min())
print("Maior data:", datas.max())

In [ ]:
# Transforma o texto em lista
import ast
steps_lista = df_original["steps"].apply(ast.literal_eval)
ingredients_lista = df_original["ingredients"].apply(ast.literal_eval)
nutrition_lista = df_original["nutrition"].apply(ast.literal_eval)
tags_lista = df_original["tags"].apply(ast.literal_eval)
print("✓ Listas prontas.")

In [ ]:
# Q13: n_steps e n_ingredients batem com o tamanho das listas?
print("n_steps não bate:", (steps_lista.apply(len) != df_original["n_steps"]).sum())
print("n_ingredients não bate:", (ingredients_lista.apply(len) != df_original["n_ingredients"]).sum())

In [ ]:
# Q14: nutrition tem sempre 7 valores? As calorias fazem sentido?
print("Receitas sem 7 valores:", (nutrition_lista.apply(len) != 7).sum())
calorias = nutrition_lista.str[0]
print("Calorias negativas:", (calorias < 0).sum())
display(calorias.describe())

In [ ]:
# Q15: alguma receita ficou sem tag?
print("Receitas sem nenhuma tag:", (tags_lista.apply(len) == 0).sum())

In [ ]:
# Cópia de trabalho: a base original fica intacta
df = df_original.copy()
print("Linhas no início:", len(df))

In [ ]:
# D1 remover receitas duplicadas
antes = len(df)
df = df.drop_duplicates(subset=["name", "contributor_id"], keep="first")
print("Linhas removidas:", antes - len(df))
print("Duplicadas restantes:", df.duplicated(subset=["name", "contributor_id"]).sum())

In [ ]:
# D2 remover receita sem nome e receita sem passos
antes = len(df)
df = df[df["name"].notna() & (df["n_steps"] > 0)]
print("Linhas removidas:", antes - len(df))
print("Sem nome restantes:", df["name"].isna().sum())
print("Sem passos restantes:", (df["n_steps"] == 0).sum())

In [ ]:
# D3 description mantida, só conferindo
print("Receitas sem description (mantidas):", df["description"].isna().sum())

In [ ]:
# D4 trocar tempos impossíveis por vazio (NaN)
df.loc[df["minutes"] == 0, "minutes"] = np.nan
df.loc[df["minutes"] == 2147483647, "minutes"] = np.nan
print("Tempos agora vazios:", df["minutes"].isna().sum())
print("Menor tempo:", df["minutes"].min())
print("Maior tempo:", df["minutes"].max())
print("Média:", round(df["minutes"].mean(), 2), "| Mediana:", df["minutes"].median())

In [ ]:
# D6 converter tipos e formatos
df["submitted"] = pd.to_datetime(df["submitted"])

for coluna in ["tags", "steps", "ingredients", "nutrition"]:
    df[coluna] = df[coluna].apply(ast.literal_eval)

colunas_nutri = ["calories", "total_fat_pdv", "sugar_pdv", "sodium_pdv", "protein_pdv", "saturated_fat_pdv", "carbohydrates_pdv"]
df[colunas_nutri] = pd.DataFrame(df["nutrition"].tolist(), index=df.index)

print("Tipo de submitted:", df["submitted"].dtype)
print("Tipo de tags:", type(df["tags"].iloc[0]))
display(df[colunas_nutri].head())

In [ ]:
# Salva a base preparada
os.makedirs("data", exist_ok=True)
CAMINHO_FINAL = "data/recipes_ra1.parquet"
df.to_parquet(CAMINHO_FINAL, index=False)


df_final = pd.read_parquet(CAMINHO_FINAL)
assert df_final.shape == (231630, 19), "A base final não é a esperada (231.630 × 19)"
print(f"✓ Arquivo salvo: {CAMINHO_FINAL}")
print(f"Shape: {df_final.shape[0]} linhas × {df_final.shape[1]} colunas")
df_final.info()

In [ ]:
# Verificações finais da base preparada
print("--- Duplicidades ---")
print("ids repetidos:", df_final["id"].duplicated().sum())
print("nome + autor repetidos:", df_final.duplicated(subset=["name", "contributor_id"]).sum())

print("\n--- Ausências ---")
display(df_final.isna().sum())

print("\n--- Domínios ---")
print("minutes igual a 0:", (df_final["minutes"] == 0).sum())
print("minutes máximo:", df_final["minutes"].max())
print("n_steps igual a 0:", (df_final["n_steps"] == 0).sum())
print("n_ingredients igual a 0:", (df_final["n_ingredients"] == 0).sum())
print("Período:", df_final["submitted"].min(), "a", df_final["submitted"].max())
print("Valores negativos por coluna numérica:")
display(df_final.select_dtypes("number").lt(0).sum())

print("\n--- Integridade ---")
print("n_steps diferente da qtd de passos:", (df_final["n_steps"] != df_final["steps"].apply(len)).sum())
print("n_ingredients diferente da qtd de ingredientes:", (df_final["n_ingredients"] != df_final["ingredients"].apply(len)).sum())